[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C09/E09-2_Ultrasons_Couts.ipynb)

# Contrôle ultrasonore et coût des erreurs : décider au moindre coût

## Le contexte

Le contrôle ultrasonore d'une soudure relève des indications : pour chacune, la sonde utilisée, l'épaisseur de la pièce, le parcours du faisceau jusqu'à l'indication, l'amplitude de l'écho et la longueur de l'indication. Une découpe suivie d'une radiographie établit ensuite si le défaut dépasse 2,5 mm, seuil de refus. Il faut décider, indication par indication, d'accepter la pièce ou de l'écarter.

Les deux erreurs ne coûtent pas pareil. Écarter à tort une bonne pièce coûte 1 unité, la valeur de la pièce. Laisser passer un défaut coûte 20 unités, avec le retour, la réparation et le risque en service. Ces coûts relatifs sont les hypothèses du service contrôle.

La grandeur à prédire est une catégorie, conforme ou non conforme : c'est un problème de classification binaire, jugé par un coût.

Défi : décider pour chaque indication et obtenir un gain de coût supérieur à celui du modèle de départ sur la base de test. Le gain de coût compare le coût moyen par pièce du modèle à celui de la meilleure politique sans apprentissage, tout accepter ou tout écarter : il vaut 0 si l'on ne fait pas mieux, 1 si l'on ne paie plus rien, et il devient négatif si l'on paie davantage. Le modèle de départ est une régression logistique au seuil habituel de 0,5 ; son gain est de -0,657, négatif : ce seuil fait ici pire que la meilleure politique sans apprentissage. À vous de faire mieux, en choisissant le seuil. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Ultrasons_Couts.xlsx`, dans le dossier `donnees` du dépôt : 500 indications, 6 colonnes. Une ligne est une indication relevée au contrôle d'une soudure.

| Colonne | Signification | Unité |
|---|---|---|
| `sonde` | sonde utilisée : `droite`, `angle_45` ou `angle_70` | catégorie |
| `epaisseur_mm` | épaisseur de la pièce | mm |
| `profondeur_mm` | parcours du faisceau jusqu'à l'indication | mm |
| `amplitude_dB` | amplitude de l'écho par rapport au trou de référence de 2 mm à 20 mm | dB |
| `longueur_indication_mm` | longueur de l'indication, mesurée à 6 dB de chute | mm |
| `non_conforme` | cible : `1` si le défaut dépasse 2,5 mm, `0` sinon | 0 ou 1 |

Origine : jeu simulé, d'après la loi d'écho d'un réflecteur en disque : l'amplitude croît avec le diamètre du défaut et décroît avec le parcours du faisceau. Quelques profondeurs et longueurs ne sont pas relevées. Il enseigne une méthode, et aucune conclusion de contrôle n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la part des défauts avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Ultrasons_Couts.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Ultrasons_Couts.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "non_conforme"
categories = ["sonde"]
mesures = ["epaisseur_mm", "profondeur_mm", "amplitude_dB", "longueur_indication_mm"]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print(df[cible].value_counts().to_dict())
df.head(3)

Lecture du résultat : 500 indications, 6 colonnes. La profondeur compte 15 cases vides et la longueur 15, qu'il faudra remplir. 105 indications correspondent à un défaut de plus de 2,5 mm, soit 21,0 % : les défauts sont minoritaires.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Ultrasons_Couts.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des indications est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié, pour que la part des défauts soit la même dans les deux bases.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "indications, dont", int(y_train.sum()), "défauts")
print("test :", len(X_test), "indications, dont", int(y_test.sum()), "défauts")

### Étape 3 : définir le coût et le gain

Le coût moyen par pièce d'une décision est la somme des coûts de ses erreurs, divisée par le nombre de pièces. Le gain de coût le compare à la meilleure politique sans apprentissage. Les deux politiques sans apprentissage sont lues sur l'apprentissage, avant de toucher au test.

In [ ]:
COUT_FAUSSE_ALERTE = 1      # une bonne pièce écartée à tort
COUT_FAUX_NEGATIF = 20       # un défaut laissé passer


def cout_moyen(y_vrai, y_predit):
    y_vrai, y_predit = np.asarray(y_vrai), np.asarray(y_predit)
    faux_negatifs = np.sum((y_vrai == 1) & (y_predit == 0))
    fausses_alertes = np.sum((y_vrai == 0) & (y_predit == 1))
    return (COUT_FAUX_NEGATIF * faux_negatifs + COUT_FAUSSE_ALERTE * fausses_alertes) / len(y_vrai)


def gain_cout(y_vrai, y_predit):
    """1 moins le coût moyen du modèle sur le coût moyen de la meilleure politique sans apprentissage."""
    tout_accepter = cout_moyen(y_vrai, np.zeros(len(y_vrai), dtype=int))
    tout_ecarter = cout_moyen(y_vrai, np.ones(len(y_vrai), dtype=int))
    return 1 - cout_moyen(y_vrai, y_predit) / min(tout_accepter, tout_ecarter)


cout_accepter = cout_moyen(y_train, np.zeros(len(y_train), dtype=int))
cout_ecarter = cout_moyen(y_train, np.ones(len(y_train), dtype=int))
seuil_theorique = COUT_FAUSSE_ALERTE / (COUT_FAUSSE_ALERTE + COUT_FAUX_NEGATIF)
print(f"coût moyen par pièce, tout accepter : {cout_accepter:.3f}")
print(f"coût moyen par pièce, tout écarter : {cout_ecarter:.3f}")
print(f"seuil de probabilité de défaut au-dessus duquel écarter : {seuil_theorique:.3f}")

Lecture du résultat : accepter toutes les pièces coûte 4,213 unités par pièce, parce que 21 % des indications sont des défauts à 20 unités chacun. Écarter toutes les pièces coûte 0,789 : c'est la meilleure politique sans apprentissage, et le gain est mesuré contre elle. Un modèle qui estime bien la probabilité de défaut doit écarter une pièce dès que cette probabilité dépasse 0,048, c'est-à-dire 1 sur 21 : écarter à tort coûte peu, laisser passer coûte cher, et le seuil de 0,5 est bien trop haut.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression logistique, qui estime la probabilité de défaut de chaque indication, puis écarte toute indication dont la probabilité dépasse 0,5. La sonde est encodée en colonnes 0 ou 1, les cases vides sont remplies par la médiane de l'apprentissage et les mesures sont mises à l'échelle, dans le même pipeline. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle décide pour des indications qu'il n'a jamais vues, avec le seuil de 0,5. On lit le gain de coût, le coût moyen par pièce du modèle et de la meilleure politique sans apprentissage, et le détail des erreurs.

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = modele.predict(X_test)
score_depart = gain_cout(y_test, y_pred)
cout_modele = cout_moyen(y_test, y_pred)
cout_trivial = min(cout_moyen(y_test, np.zeros(len(y_test), dtype=int)), cout_moyen(y_test, np.ones(len(y_test), dtype=int)))
vrais_negatifs, fausses_alertes, defauts_manques, defauts_trouves = confusion_matrix(y_test, y_pred).ravel()
print(f"gain de coût : {score_depart:.3f}")
print(f"coût moyen par pièce : {cout_modele:.3f}, contre {cout_trivial:.3f} pour la meilleure politique sans apprentissage")
print(f"défauts trouvés : {defauts_trouves}, défauts laissés passer : {defauts_manques}, bonnes pièces écartées : {fausses_alertes}")

Lecture du résultat : le gain de coût est de -0,657, négatif : le modèle coûte 1,312 unités par pièce, plus que les 0,792 d'une politique qui écarte tout sans rien apprendre. Il a laissé passer 8 défauts sur 26, à 20 unités chacun, pour 4 bonnes pièces écartées à tort. Il ne se trompe pas beaucoup ; il se trompe du mauvais côté. C'est ce chiffre, mesuré sur des indications que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : le seuil de moindre coût

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les trois cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première obtient des probabilités de défaut hors échantillon sur l'apprentissage, par validation croisée, pour quatre modèles, et cherche pour chacun le seuil de coût moyen minimal. L'un d'eux est une régression logistique à laquelle on donne le logarithme de la profondeur : l'amplitude d'un écho décroît avec le logarithme du parcours du faisceau, et la frontière entre défauts à refuser et à accepter devient presque une droite dans les mesures ainsi transformées. La deuxième trace le coût en fonction du seuil pour le modèle retenu. La troisième mesure ce modèle, avec ce seuil, une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : le modèle retenu par la validation croisée, logistique sur le logarithme de la profondeur, au seuil de 0,20, coûte 0,120 unité par pièce sur le test, contre 0,792 pour la meilleure politique sans apprentissage : le gain de coût est de 0,848. Il dépasse de 1,505 le modèle de départ, dont le seuil de 0,5 fait ici pire que de ne rien apprendre. Le test compte 26 défauts à 20 unités : un défaut de plus ou de moins laissé passer déplace le gain de 0,202, et l'écart entre deux modèles proches ne s'y lit pas.

## Le score de référence

Le score publié est le gain de coût du modèle de départ de l'étape 4, au seuil de 0,5, sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = gain_cout(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE gain_cout = %.3f" % score)

## Synthèse

1. Quand les deux erreurs ne coûtent pas pareil, la précision et l'exactitude ne suffisent pas : on chiffre chaque décision en coût, et on compare à la meilleure politique sans apprentissage.
2. Pour une probabilité bien estimée, le seuil de moindre coût est le coût d'une fausse alerte divisé par la somme des deux coûts ; ici il vaut 0,048, très inférieur à 0,5.
3. Le seuil se choisit sur des probabilités hors échantillon de l'apprentissage, par validation croisée, en minimisant le coût moyen ; il ne se règle jamais sur le test.
4. Un seuil très bas demande des probabilités fiables dans les faibles valeurs : en validation croisée, le coût moyen minimal est de 0,283 pour la régression logistique, 0,184 pour la même régression sur le logarithme de la profondeur, 0,477 pour la forêt et 0,224 pour le gradient boosting. Mettre la physique dans les variables, ici le logarithme de la profondeur, peut valoir davantage que changer de modèle.
5. Le test ne sert qu'une fois, au modèle retenu et à son seuil ; un gain négatif signifie qu'un modèle sans seuil adapté coûte plus cher que de ne rien apprendre.